In [ ]:
from plant_tokenizer import SOS_TOKEN, EOS_TOKEN, PAD_TOKEN, META_TOKEN
real_image_data_dir = "../real_images"

image_list = os.listdir(real_image_data_dir)
thr = 0.5
n_figures = len(image_list)
background_color = [204, 204, 204]  # RGB 값 (204, 204, 204)

data_list = []
for i, image_name in enumerate(image_list):
    data = dict()
    img = cv2.imread(os.path.join(real_image_data_dir, image_name))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)

    # Calc ExG
    # Convert to float
    img_VI = img.astype(np.float32)

    # Calculate ExG
    rgb_sum = img_VI[:, :, 0] + img_VI[:, :, 1] + img_VI[:, :, 2]
    r = img_VI[:, :, 0] 
    g = img_VI[:, :, 1] 
    b = img_VI[:, :, 2]

    ExG = (2*g - r - b) / (rgb_sum+0.0001)

    # Calculate the threshold
    threshold = np.mean(ExG) + thr*np.std(ExG)
    # Threshold the image
    mask = ExG > threshold
    
    # 소금-후추 노이즈 제거를 위한 중앙값 필터 적용
    mask = mask.astype(np.uint8) * 255
    mask = cv2.medianBlur(mask, 5)
    
    # 모폴로지 연산을 통한 마스크 개선
    # 작은 구멍 제거를 위한 닫기 연산 (dilate 후 erode)
    kernel = np.ones((5,5), np.uint8)
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, kernel)
    
    # 작은 돌기 제거를 위한 열기 연산 (erode 후 dilate)
    mask = cv2.morphologyEx(mask, cv2.MORPH_OPEN, kernel)
    
    # 이진 마스크로 변환
    mask = mask > 0
    
    # 3차원 마스크 생성 (RGB 채널에 적용하기 위함)
    mask_3d = np.stack([mask, mask, mask], axis=2)
    
    # 배경색을 가진 빈 이미지 생성
    bg_removed_img = np.ones_like(img) * background_color
    
    # 마스크를 이용해 원본 이미지에서 식물 부분만 유지하고 배경은 지정된 색상으로 대체
    result_img = np.where(mask_3d, img, bg_removed_img)

    # 결과 이미지 출력 (원본, 마스크, 배경 제거된 이미지)
    plt.figure(figsize=(15, 5))
    
    plt.subplot(1, 4, 1)
    plt.imshow(img)
    plt.title('Original Image')
    plt.axis(False)
    
    plt.subplot(1, 4, 2)
    plt.imshow(mask, cmap='gray')
    plt.title('Vegetation Mask')
    plt.axis(False)
    
    plt.subplot(1, 4, 3)
    plt.imshow(result_img)
    plt.title('Background Removed')
    plt.axis(False)
    
    
    # 결과 이미지 저장 (선택적)
    # result_path = os.path.join(real_image_data_dir, f"processed_{image_name}")
    # cv2.imwrite(result_path, cv2.cvtColor(result_img.astype(np.uint8), cv2.COLOR_RGB2BGR))

    leaf_area, plant_width, plant_height, leaf_img, _ = process_leaf_image(result_img.astype(np.uint8), sqaure_crop=True, thr=0.0)
    
    if 1:
        #leaf_img = cv2.resize(leaf_img, (image_size, image_size))
        leaf_img = cv2.resize(result_img.astype(np.uint8), (image_size, image_size))
    else:
        # Simulating side view?
        leaf_img = cv2.resize(leaf_img, (image_size//2, image_size//2))
        canvas = np.ones((image_size, image_size,3),np.uint8) * background_color
        canvas[:image_size//2,:image_size//2] = leaf_img
        canvas[:image_size//2,image_size//2:] = leaf_img
        canvas[image_size//2:,image_size//2:] = leaf_img
        canvas[image_size//2:,:image_size//2] = leaf_img
        leaf_img = canvas

    plant_info = [leaf_area, plant_width, plant_height]
    
    # Make a dummy vector for plant_info
    plant_info_vec = np.concatenate(([0,0], plant_info))
    # Tokenize plant info
    plant_info_token = vec2token([plant_info_vec])
    plant_info_token = np.concatenate(([META_TOKEN], plant_info_token[1:].astype('int64'), [META_TOKEN]))
    data["pixel_values"] = image_processor(images=leaf_img, return_tensors="pt").pixel_values[0]
    data["plant_info"] = plant_info_token

    data_list.append(data)

    plt.subplot(1, 4, 4)
    plt.imshow(leaf_img)
    plt.title('Resized & Croped')
    plt.axis(False)

    plt.tight_layout()
    plt.show()

In [ ]:
# Prepare the figure
n_figures = 5
fig, axes = plt.subplots(2, n_figures, figsize=(20, 8))

# Create temp folder
temp_folder = "temp"
shutil.rmtree(temp_folder, ignore_errors=True)
os.makedirs(temp_folder, exist_ok=True)

device = model.device
for idx, data in enumerate(data_list):
    if idx >= n_figures:
        break
    image = data["pixel_values"]
    plant_info = data["plant_info"]
    

    if image.dim() == 3:
        image = image.unsqueeze(0)

    image = image.to(device)

    image_vis = image[0].permute(1, 2, 0).cpu()
    image_vis = cv2.normalize(np.array(image_vis), None, alpha=0, beta=255, norm_type=cv2.NORM_MINMAX, dtype=cv2.CV_8U)
    
    ############## Generate
    with torch.no_grad():
        plant_info = torch.tensor(plant_info, dtype=torch.long).unsqueeze(0).to(model.device)  # Ensure plant_info is a tens
        result = model.generate(image,
                                decoder_start_token_id=SOS_TOKEN,
                                decoder_input_ids=plant_info,
                                eos_token_id=EOS_TOKEN,
                                pad_token_id=PAD_TOKEN,
                                # do_sample=True,
                                # num_beams=5,
                                max_length=max_length,
                                use_cache=True
                                )
        result = result.squeeze().cpu().numpy()[6:]
    try:
        plant_vec = token2vec(result)
        plant_xml = vec2xml(plant_vec)
        plant_xml_file_name = f"temp/plant_{idx}_est.xml"
        plant_xml = recursive_to_linked(plant_xml)
        plant_xml_str = pretty_print_xml(plant_xml)
        with open(plant_xml_file_name, "w") as f:
            f.write(plant_xml_str)

        re_render_xml(os.path.abspath(temp_folder), os.path.abspath(plant_xml_file_name))

        if side_view:
            img, _ = load_sideview_images(temp_folder, plant_xml_file_name.replace("xml","jpeg"), image_size, True)
        else:
            img = cv2.imread(plant_xml_file_name.replace("xml","jpeg"))
            img = cv2.cvtColor(img, cv2.COLOR_RGB2BGR)
            leaf_area, plant_width, plant_height, leaf_img, _ = process_leaf_image(gt_img, sqaure_crop=True, thr=0.0)
            img = cv2.resize(leaf_img, (image_size, image_size))
    except:
        text = "Error"
        font = cv2.FONT_HERSHEY_SIMPLEX
        img = np.zeros_like(image_vis)
        text_size = cv2.getTextSize(text, font, 1, 2)[0]
        text_x = (img.shape[1] - text_size[0]) // 2
        text_y = (img.shape[0] + text_size[1]) // 2
        cv2.putText(img, text, (text_x, text_y), font, 1, (0, 0, 255), 2)


    row, col = divmod(idx, n_figures)

    image_vis = cv2.cvtColor(image_vis, cv2.COLOR_RGB2BGR)
    axes[row, col].imshow(image_vis)
    axes[row, col].set_title(f"Input Image {idx + 1}")
    axes[row, col].axis('off')

    
    axes[row+1, col].imshow(img)
    axes[row+1, col].set_title(f"Output Model {idx + 1}")
    axes[row+1, col].axis('off')

plt.tight_layout()
plt.show()